In [ ]:
import pandas as pd

df = pd.read_csv("Telco.csv")
df_org = df.copy()
print(df.head())

In [ ]:
df = df.drop(columns = ['customerID'])

In [ ]:
print(df['gender'].unique())

In [ ]:
df['gender'] = df['gender'].map({"Male": 0, "Female": 1})

In [ ]:
print(df['SeniorCitizen'].value_counts())

In [ ]:
print(df['Partner'].value_counts())

In [ ]:
df['Partner'] = df['Partner'].map({"Yes": 1, "No": 0})

In [ ]:
print(df['Dependents'].value_counts())

In [ ]:
df['Dependents'] = df['Dependents'].map({"Yes":1, "No": 0})

In [ ]:
print(df['PhoneService'].value_counts())

In [ ]:
df['PhoneService'] = df['PhoneService'].map({'Yes': 1, "No": 0})

In [ ]:
print(df['MultipleLines'].value_counts())

it's category so we OHE this ;p

In [ ]:
print(df['InternetService'].unique())

In [ ]:
print(df['OnlineSecurity'].unique())

In [ ]:
print(df.head())

In [ ]:
print(df['OnlineBackup'].unique())

In [ ]:
print(df['DeviceProtection'].unique())

In [ ]:
print(df['TechSupport'].unique())

In [ ]:
print(df['StreamingTV'].unique())

In [ ]:
print(df['StreamingMovies'].unique())

In [ ]:
print(df['Contract'].unique())

In [ ]:
print(df['PaperlessBilling'].unique())

In [ ]:
df['PaperlessBilling'] = df['PaperlessBilling'].map({"Yes": 1, "No": 0})

In [ ]:
print(df['PaymentMethod'].unique())

In [ ]:
df = df.apply(lambda x: x.str.strip() if x.dtype == 'object' else x)

should've done this at the beginning ;-;

In [ ]:
print(df['Churn'].value_counts())

In [ ]:
df['Churn'] = df['Churn'].map({"Yes": 1, "No": 0})

In [ ]:
print(df.info())

In [ ]:
pd.to_numeric(df['TotalCharges'], errors = 'coerce').isna().sum()

In [ ]:
import numpy as np

df['TotalCharges'] = df['TotalCharges'].replace(' ', np.nan)

In [ ]:
df['TotalCharges'] = df["TotalCharges"].astype(float)

In [ ]:
print(df.info())

In [ ]:
from sklearn.model_selection import train_test_split

df_org1 = df.copy()

X = df.drop(columns=['Churn'])
y = df['Churn']

X_train , X_test , y_train, y_test = train_test_split(X, y, test_size=0.2 , random_state= 42)


In [ ]:
mean_val = X_train['TotalCharges'].mean()
X_train["TotalCharges"] = X_train['TotalCharges'].fillna(mean_val)
X_test["TotalCharges"] = X_test['TotalCharges'].fillna(mean_val)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

cols_scale = ['tenure', 'MonthlyCharges', 'TotalCharges']
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[cols_scale] = scaler.fit_transform(X_train_scaled[cols_scale])
X_test_scaled[cols_scale] = scaler.transform(X_test_scaled[cols_scale])

In [ ]:
print(cols_scale)

In [ ]:
col_OHE =  X_train_scaled.select_dtypes(include= 'str').columns

X_train_scaled = pd.get_dummies(X_train_scaled, columns= col_OHE, drop_first=True)
X_test_scaled = pd.get_dummies(X_test_scaled, columns= col_OHE, drop_first=True)

X_train_scaled, X_test_scaled = X_train_scaled.align(X_test_scaled, join = 'left', axis=1, fill_value=0)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

models = {
    "Logistic Regression": LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced'),
    "Decision Tree": DecisionTreeClassifier(random_state=42, class_weight='balanced'), 
    "Random Forest": RandomForestClassifier(random_state=42,  class_weight= 'balanced')
}

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    test_acc = accuracy_score(y_test, model.predict(X_test_scaled))
    train_acc = accuracy_score(y_train, model.predict(X_train_scaled))
    print(f"{name}: Test = {test_acc}, Train = {train_acc}")

In [ ]:
from sklearn.metrics  import classification_report

for name, model in models.items():
    print(name)
    print(classification_report(y_test, model.predict(X_test_scaled)))


In [ ]:
from sklearn.model_selection import cross_val_score


for name, model in models.items():
    scores = cross_val_score(model, X_train_scaled, y_train, cv = 5)
    print(f"{name}: {scores.mean()} +/- {scores.std()}")

In [ ]:
lr_param_grid = {
    'C': [0.01, 0.1, 1, 10, 100],
    "solver": ["lbfgs", "liblinear"], 
}

In [ ]:
from sklearn.model_selection import GridSearchCV

lr_grid = GridSearchCV(
    LogisticRegression(random_state=42, max_iter= 1000, class_weight= 'balanced'),
    lr_param_grid,
    cv = 5,
    n_jobs= -1, 
    scoring = 'recall'
)

lr_grid.fit(X_train_scaled, y_train)
print(lr_grid.best_params_)
print(lr_grid.best_score_)
best_model = lr_grid.best_estimator_
print(best_model.score(X_test_scaled, y_test))

In [ ]:
y_pred = best_model.predict(X_test_scaled)
print(classification_report(y_test, y_pred))

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

cm = confusion_matrix(y_test,  y_pred)
sns.heatmap(cm, annot = True, fmt = 'd', xticklabels= ["No", "Yes"], yticklabels=["No", "Yes"] )
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion matrix")
plt.show()

In [ ]:
import joblib
joblib.dump(best_model, "Telco.pkl")

In [ ]:
loaded_model = joblib.load("Telco.pkl")
print(loaded_model.score(X_test_scaled, y_test)) 

MODEL SUCCESSFULLY LOADED